# Geometry Optimization Workflow

This notebook submits a geometry optimization using SIESTA via AiiDA.

## Initialization

In [1]:
# AiiDA imports.
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: default.

In [2]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [3]:
# General imports.
import urllib.parse as urlparse
import yaml
from copy import deepcopy
import importlib.resources
from collections import defaultdict

import ipywidgets as ipw
from IPython.display import clear_output


# AiiDAlab imports.
import aiidalab_widgets_base as awb
import aiidalab_widgets_empa as awe
from aiida import orm, plugins


# Custom imports.
from empasiesta_tools.widgets import editors, inputs


#Not required by AiiDA
import os
import os.path as op
import sys

import json

#AiiDA classes and functions
from aiida.engine import submit
from aiida.orm import Dict, KpointsData, StructureData, load_code
from aiida.tools.data.array.kpoints.legacy import get_explicit_kpoints_path as legacy_path
from aiida.tools import get_explicit_kpoints_path
from aiida_pseudo.data.pseudo.psf import PsfData
from aiida_siesta.workflows.base import SiestaBaseWorkChain

<IPython.core.display.Javascript object>

/opt/conda/lib/python3.9/site-packages/aiida/plugins/entry_point.py:351: AiidaDeprecationWarning: The entry point `array.trajectory` is deprecated. Please replace it with `core.array.trajectory`. (this will be removed in v3)
  warn_deprecation(f'The entry point `{name}` is deprecated. Please replace it with `core.{name}`.', version=3)


## Structure Selection and Protocol Editing

In [4]:
os.environ["AIIDA_SIESTA_PROTOCOLS"] = "/home/jovyan/opt/aiidalab-empa-siesta/protocols/myprotocols.yaml"

In [5]:
# with importlib.resources.files("aiida_siesta.utils.protocols_system") \
#         .joinpath("protocols_registry.yaml").open("r") as handle:
#     data = yaml.safe_load(handle)

yaml_path = "/home/jovyan/opt/aiidalab-empa-siesta/protocols/protocols_registry.yaml"

with open(yaml_path, "r") as handle:
    data = yaml.safe_load(handle)
    
protocol_names = list(data.keys())

In [6]:
def parse_band_lines(text):
    """
    Parse band-lines textarea content into a list of segments like:
    [
        (label1, (x1,y1,z1), label2, (x2,y2,z2), npoints),
        ...
    ]
    """
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    parsed = []

    # parse all points first
    pts = []
    for line in lines:
        parts = line.split()
        # last token = label (e.g. Γ)
        label = parts[-1]
        npoints = int(parts[-2])
        coords = tuple(float(v) for v in parts[:-2])
        pts.append((label, coords, npoints))

    # convert to segments
    kpp = []
    for i in range(len(pts) - 1):
        l1, c1, n1 = pts[i]
        l2, c2, n2 = pts[i+1]

        # important: each segment uses the number of points of the *next* line
        kpp.append((l1, c1, l2, c2, n2))

    return kpp

In [7]:
def write_updated_protocol():
    # --------------------------------------------
    # Collect updated dictionary from accordion
    # --------------------------------------------
    updated = collect_widget_values()

    # Ensure "parameters" exists
    params = updated.setdefault("parameters", {})

    # -----------------------------------------------------------
    # FIX KPOINTS VALUES (convert strings → float/list-of-floats)
    # -----------------------------------------------------------
    if "kpoints" in updated:
        kp = updated["kpoints"]

        # distance: convert string → float
        if "distance" in kp:
            kp["distance"] = float(kp["distance"])

        # offset: convert string → list of floats
        if "offset" in kp:
            val = kp["offset"]
            if isinstance(val, str):
                cleaned = val.replace("[", "").replace("]", "").replace(",", " ")
                kp["offset"] = [float(x) for x in cleaned.split()]

    # -----------------------------------------------------------
    # Inject BANDS parameters
    # -----------------------------------------------------------
#     if bands_checkbox.value:

#         # Flat boolean parameters
#         params["BandsSave"] = str(bands_saves_checkbox.value).lower()
#         params["WriteBands"] = str(bands_writebands_checkbox.value).lower()
#         params["BandLinesScale"] = bands_scale_dropdown.value

#         # Block with band lines
#         params["%block band-lines"] = (
#             f"\n{bands_textarea.value}\n%endblock band-lines"
#         )

    # -----------------------------------------------------------
    # Inject LDOS / PDOS blocks into parameters
    # -----------------------------------------------------------
    if ldos_checkbox.value:
        params["%block local-density-of-states"] = (
            f"\n{ldos_text.value}\n%endblock local-density-of-states"
        )

    if pdos_checkbox.value:
        params["%block projected-density-of-states"] = (
            f"\n{pdos_text.value}\n%endblock projected-density-of-states"
        )

    # -----------------------------------------------------------
    # Wrap under "submitted_protocol" and write file
    # -----------------------------------------------------------
    final_yaml = {"submitted_protocol": updated}

    output_path = "/home/jovyan/opt/aiidalab-empa-siesta/protocols/myprotocols.yaml"

    with open(output_path, "w") as handle:
        yaml.dump(
            final_yaml,
            handle,
            sort_keys=False,
            default_flow_style=False
        )

    print("✔ Protocol written to:", output_path)


In [8]:
def nested_dict():
    """Helper to create arbitrarily nested dictionaries."""
    return defaultdict(nested_dict)

def set_nested(dictionary, path, value):
    """Set a nested dictionary value given a dot-separated path."""
    keys = path.split(".")
    d = dictionary
    for k in keys[:-1]:
        d = d[k]
    d[keys[-1]] = value

def collect_widget_values():
    """
    Read all widget values from the accordion and rebuild the updated protocol
    as a nested dictionary suitable for YAML serialization.
    All values except JSON lists are exported as strings.
    """
    result = {'description':'automatically generated'}

    if not accordion.children:
        raise RuntimeError("Accordion has no children; select a structure first.")

    # The single accordion contains one Tab
    tabs = accordion.children[0]

    for tab_index, box in enumerate(tabs.children):
        section_name = tabs.get_title(tab_index)

        # Extract description names of all widgets in this section
        keypaths = [w.description for w in box.children]

        # ------------------------------------------------------------
        # CASE 1: section contains nested fields (normal dict sections)
        # ------------------------------------------------------------
        if any(k.strip() for k in keypaths):
            section_dict = nested_dict()

            for w in box.children:
                keypath = w.description

                # LISTS (Textarea with JSON content)
                if isinstance(w, ipw.Textarea):
                    # Try to decode JSON list
                    try:
                        value = json.loads(w.value)      # correct YAML list
                    except Exception:
                        value = str(w.value)             # fallback

                # EVERYTHING ELSE → STRING
                else:
                    value = str(w.value)

                # Insert into nested dictionary
                set_nested(section_dict, keypath, value)

            # convert defaultdict into normal dict
            import json as _json
            section_dict = yaml.safe_load(
                yaml.safe_dump(_json.loads(_json.dumps(section_dict)))
            )
            result[section_name] = section_dict

        # ------------------------------------------------------------
        # CASE 2: scalar section (e.g., pseudo_family)
        # ------------------------------------------------------------
        else:
            # section has exactly one widget, its value is scalar
            w = box.children[0]
            section_value = str(w.value)
            result[section_name] = section_value

    return result


In [9]:
# Structure selector.
build_slab = editors.BuildSlab(title="Build slab")
input_details = inputs.InputDetails()

structure_selector = awb.StructureManagerWidget(
    importers=[
        awb.StructureUploadWidget(title="Import from computer"),
        awb.StructureBrowserWidget(title="AiiDA database"),
        awb.SmilesWidget(title="From SMILES"),
        awe.CdxmlUploadWidget(title="CDXML"),
    ],
    editors=[
        awb.BasicStructureEditor(title="Edit structure"),
        build_slab,
        awb.BasicCellEditor(),
        editors.InsertStructureWidget(title="Insert molecule"),
    ],
    storable=True,
    node_class="StructureData",
)
ipw.dlink((structure_selector, "structure"), (build_slab, "molecule"))
ipw.dlink((structure_selector, "structure"), (input_details, "structure"))
ipw.dlink((input_details, "details"), (build_slab, "details"))
display(structure_selector)

# Code.
code_input_widget = awb.ComputationalResourcesWidget(
    description="SIESTA code:", default_calc_job_plugin="siesta"
)
resources = awe.ProcessResourcesWidget()

output = ipw.Output()

StructureManagerWidget(children=(Accordion(children=(Tab(children=(StructureUploadWidget(children=(FileUpload(…

/home/jovyan/.local/lib/python3.9/site-packages/aiidalab_widgets_base/viewers.py:737: DeprecationWarning: dict interface is deprecated. Use attribute interface instead
  self.cell_spacegroup.value = f"Spacegroup: {symmetry_dataset['international']} (No.{symmetry_dataset['number']})"


In [10]:
url = urlparse.urlsplit(jupyter_notebook_url)
parsed_url = urlparse.parse_qs(url.query)
if "structure_uuid" in parsed_url:
    structure_selector.input_structure = load_node(parsed_url["structure_uuid"][0])

In [11]:


# -------------------------------------------------------------
# Utility: get element symbols from structure
# -------------------------------------------------------------
def get_elements_from_structure(structure_node):
    try:
        return sorted({site.kind_name for site in structure_node.sites})
    except Exception:
        return []

def make_widget(fullname, value):
    """
    Create widgets that NEVER coerce integers into floats.
    All non-boolean and non-list values → Text widget.
    """

    # Boolean → Checkbox
    if isinstance(value, bool):
        w = ipw.Checkbox(description=fullname, value=value)

    # List → Textarea with JSON
    elif isinstance(value, list):
        w = ipw.Textarea(description=fullname, value=json.dumps(value))

    # EVERYTHING ELSE → Text (preserve exact formatting!)
    else:
        # Convert original value to string exactly as it appeared in YAML
        w = ipw.Text(description=fullname, value=str(value))

    # ---------------------------------------------------------
    # 🎨 STYLE IMPROVEMENTS FOR READABILITY
    # ---------------------------------------------------------

    # Wider text field
    w.layout = ipw.Layout(
        width="450px",         # value area
        min_width="400px",
        max_width="600px",
    )

    # Wider description label (key)
    w.style = {
        "description_width": "200px"   # FINALLY the key is readable!
    }

    # ---------------------------------------------------------
    # Track user changes
    # ---------------------------------------------------------
    w._original_value = value
    w._user_modified = False

    def handler(change, widget=w):
        if change["name"] == "value":
            widget._user_modified = True

    w.observe(handler)

    return w



# -------------------------------------------------------------
# Turn a nested dict into a list of widgets
# -------------------------------------------------------------
def widgets_from_dict(dct, prefix=""):
    widgets = []
    for key, val in dct.items():
        fullname = f"{prefix}{key}" if prefix else key

        if isinstance(val, dict):
            widgets.extend(widgets_from_dict(val, prefix=f"{fullname}."))
        else:
            widgets.append(make_widget(fullname, val))

    return widgets

# -------------------------------------------------------------
# Convert section into VBox (with optional element filtering)
# -------------------------------------------------------------
def section_to_vbox(section_data, allowed_elements=None):
    if isinstance(section_data, dict):
        if allowed_elements is not None:
            # filter atomic_heuristics
            section_data = {
                el: section_data[el]
                for el in section_data.keys()
                if el in allowed_elements
            }
        widgets = widgets_from_dict(section_data)
        return ipw.VBox(widgets)

    # scalar section (e.g. pseudo_family)
    w = make_widget("", section_data)
    return ipw.VBox([w])


#############

# ------------------------------------------------------------
# RELAX MODE WIDGETS
# ------------------------------------------------------------

relax_mode = ipw.ToggleButtons(
    options=[
        ("No relax", None),
        ("Atom relax", "atoms_only"),
        ("Full relax", "variable_cell"),
        ("Constant volume","constant_volume")
    ],
    value=None,
    description="Relax mode:",
    style={"description_width": "initial"},
)
# Force selection to "No relax"
relax_mode.value = None

relax_algorithm = ipw.Dropdown(
    options=["CG", "BROYDEN", "FIRE"],
    value="CG",
    description="Algorithm:",
    style={"description_width": "initial"},
)

relax_box = ipw.VBox([relax_mode])

def update_relax_box(change=None):
    if relax_mode.value == "no":
        relax_box.children = [relax_mode]
    else:
        relax_box.children = [relax_mode, relax_algorithm]

relax_mode.observe(update_relax_box, names="value")
update_relax_box()

# ------------------------------------------------------------
# BANDS widgets
# ------------------------------------------------------------

bands_checkbox = ipw.Checkbox(
    description="Bands",
    value=False,
)

bands_saves_checkbox = ipw.Checkbox(
    description="BandsSave",
    value=True,
)

bands_writebands_checkbox = ipw.Checkbox(
    description="WriteBands",
    value=True,
)

bands_scale_dropdown = ipw.Dropdown(
    description="BandLinesScale",
    options=[
        ("ReciprocalLatticeVectors", "ReciprocalLatticeVectors"),
    ],
    value="ReciprocalLatticeVectors",
    style={"description_width": "120px"},
)

bands_textarea = ipw.Textarea(
    description="Band lines:",
    value=(
        " 0.0  0.0  0.0    0   Γ\n"
        " 0.5  0.0  0.5   30   X\n"
        " 0.5  0.25 0.75  30   W\n"
        " 0.5  0.5  0.5   30   L\n"
        " 0.0  0.0  0.0   30   Γ"
    ),
    layout=ipw.Layout(width="450px", height="150px"),
    style={"description_width": "120px"},
)

def update_bands_box(*args):
    widgets = [bands_checkbox]

    if bands_checkbox.value:
        widgets.append(bands_saves_checkbox)
        widgets.append(bands_writebands_checkbox)
        widgets.append(bands_scale_dropdown)
        widgets.append(bands_textarea)

    bands_box.children = widgets

bands_box = ipw.VBox([])
bands_checkbox.observe(update_bands_box, "value")
update_bands_box()


# ------------------------------------------------------------
# LDOS & PDOS widgets
# ------------------------------------------------------------

pdos_checkbox = ipw.Checkbox(
    description="PDOS",
    value=False,
)

ldos_checkbox = ipw.Checkbox(
    description="LDOS",
    value=False,
)

pdos_text = ipw.Textarea(
    description="PDOS:",
    value="-6.0 -1.5 0.05 350 eV",
    layout=ipw.Layout(width="400px"),
    style={"description_width": "80px"},
)

ldos_text = ipw.Textarea(
    description="LDOS:",
    value="-4.06 -3.66 eV",
    layout=ipw.Layout(width="400px"),
    style={"description_width": "80px"},
)

# container showing only when enabled
def update_dos_boxes(*args):
    boxes = [pdos_checkbox, ldos_checkbox]

    if pdos_checkbox.value:
        boxes.append(pdos_text)
    if ldos_checkbox.value:
        boxes.append(ldos_text)

    dos_box.children = boxes

dos_box = ipw.VBox([])
pdos_checkbox.observe(update_dos_boxes, "value")
ldos_checkbox.observe(update_dos_boxes, "value")
update_dos_boxes()



# ------------------------------------------------------------
# APPLY RELAX MODES TO THE PROTOCOL BEFORE BUILDING WIDGETS
# ------------------------------------------------------------
def apply_relax_mode(proto):
    """Return a modified copy of the protocol according to relax_mode."""
    mode = relax_mode.value
    algo = relax_algorithm.value.upper()

    if mode == "no":
        return proto  # unchanged

    relax_sec = proto.setdefault("relax_additions", {})

    # Common to atom + full relax
    relax_sec['md-use-save-xv'] = '.true.'
    relax_sec["md-max-force-tol"] = "0.04 eV/ang"
    relax_sec["md-max-stress-tol"] = "0.1 GPa"
    relax_sec["md-ype-of-run"] = algo
    relax_sec["md-num-cg-steps"] = "100"
    relax_sec["md-max-cg-displ"] = "0.120 Ang"
    relax_sec["write-md-xmol"] = ".true."

    if algo == "FIRE":
        relax_sec["md-fire-time-step"] = "1.50 fs"

    if mode == "full":
        relax_sec["md-variable-cell"] = ".true."

    return proto

#############

# -------------------------------------------------------------
# The main accordion update
# -------------------------------------------------------------
accordion = ipw.Accordion(children=[])

def update_accordion(*args):
    struct = getattr(structure_selector, "structure_node", None)

    if struct is None:
        accordion.children = []
        return

    protocol_key = protocol.value
    
    # CLONE + APPLY RELAX MODE HERE
    proto = deepcopy(data[protocol_key])
    proto = apply_relax_mode(proto)

    elements = get_elements_from_structure(struct)

    tab_children = []
    tab_titles = []

    for section_name, section_content in proto.items():

        # skip YAML description
        if section_name == "description":
            continue

        # atomic heuristics → keep only present species
        if section_name == "atomic_heuristics":
            box = section_to_vbox(section_content, allowed_elements=elements)
            if not box.children:
                continue
        else:
            box = section_to_vbox(section_content)

        tab_children.append(box)
        tab_titles.append(section_name)

    tabs = ipw.Tab(children=tab_children)
    for i, title in enumerate(tab_titles):
        tabs.set_title(i, title)

    accordion.children = [tabs]
    accordion.set_title(0, "Advanced settings")

# Observe relax mode and algorithm
relax_mode.observe(update_accordion, names="value")
relax_algorithm.observe(update_accordion, names="value")
#
    
    
# -------------------------------------------------------------
# Protocol dropdown
# -------------------------------------------------------------
protocol = ipw.Dropdown(
    value=protocol_names[0],
    options=[(p, p) for p in protocol_names],
    description="Protocol:",
    style={"description_width": "initial"},
)

protocol.observe(update_accordion, names="value")
structure_selector.observe(update_accordion, names="structure_node")




In [12]:
workflow_description = ipw.Text(
    description="Workflow description:",
    placeholder="Provide the description here.",
    style={"description_width": "initial"},
    layout={"width": "70%"},
)

In [13]:
ipw.dlink((code_input_widget, "value"), (input_details, "selected_code"))


def prepare_geometry_optimization():
    with output:
        clear_output()
    if not structure_selector.structure_node:
        can_submit, msg = False, "Select a structure first."
    elif not code_input_widget.value:
        can_submit, msg = False, "Select SIESTA code."
    else:
        write_updated_protocol()
        calc_engines = {
            'siesta': {
                'code': code_input_widget.value,
                'options': {
                    'resources': {
                        'num_machines': resources.nodes, 
                        'num_mpiprocs_per_machine': resources.tasks_per_node,
                        'num_cores_per_mpiproc': resources.threads_per_task,
                    },
             "max_wallclock_seconds": resources.walltime_seconds, #'queue_name': 'DevQ', 'withmpi': True, 'account': "tcphy113c"
         }}}
        inp_gen = SiestaBaseWorkChain.inputs_generator()
        bands_path_generator = None
        builder = inp_gen.get_filled_builder(structure_selector.structure_node, calc_engines, 'submitted_protocol',bands_path_generator=bands_path_generator, relaxation_type=relax_mode.value, spin=None)
        builder.metadata.description = workflow_description.value
        if bands_checkbox.value:
            kpp = parse_band_lines(bands_textarea.value)
            tmp = legacy_path(kpp)

            bandskpoints = KpointsData()
            bandskpoints.set_kpoints(tmp[3])
            bandskpoints.labels = tmp[4]

            builder.bandskpoints = bandskpoints
        builder.metadata.label = "SIESTA_calc"

        can_submit=True
        
    if not can_submit:
        with output:
            print(msg)
            return


    return builder

In [14]:
btn_submit = awb.SubmitButtonWidget(
    SiestaBaseWorkChain,
    inputs_generator=prepare_geometry_optimization,
    disable_after_submit=False,
    append_output=True,
)

# Inputs

In [15]:
#display(protocol)
ui = ipw.VBox([protocol, relax_box,dos_box,bands_box,accordion])
ui

# Code and resources

In [16]:
display(code_input_widget, resources)

ComputationalResourcesWidget(children=(HBox(children=(Dropdown(description='SIESTA code:', options=(('siesta@l…

ProcessResourcesWidget(children=(IntText(value=2, description='# Nodes', layout=Layout(width='200px'), style=D…

# Submit

## Submission

In [17]:
display(workflow_description, btn_submit, output)

Text(value='', description='Workflow description:', layout=Layout(width='70%'), placeholder='Provide the descr…

SubmitButtonWidget(children=(Button(description='Submit', style=ButtonStyle()), HTML(value='')))

Output()